# Ingestión Manual de Datos (PDFs y Requisitos)
Este cuaderno permite ingestar directamente a la base de datos (PostgreSQL/pgvector) los requerimientos etiquetados del dataset de prueba y procesar la norma ISO 29148 desde su PDF, dividiéndola en fragmentos (chunks) de texto de tamaño configurado.

## 1. Configuración Inicial

In [15]:
import sys
import os
import time
import pandas as pd
from dotenv import load_dotenv
import warnings

warnings.filterwarnings('ignore')

load_dotenv('.env')
sys.path.append(os.path.abspath('../backend'))

# Forzar el uso de localhost para LM Studio ya que el notebook corre fuera de Docker
os.environ["LOCAL_MODELS_API"] = "http://localhost:1234/v1/"

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pdfplumber

from app.services.retriever.retriever_service import get_embeddings
from app.db.session import SessionLocal
from app.db.models import Document as DBDocument
from app.db.models import Requirement as DBRequirement

In [16]:
def embed_with_retry(embeddings_model, texts, max_retries=5, initial_delay=5.0):
    import time
    import re
    delay = initial_delay
    for attempt in range(max_retries):
        try:
            # Delay base preventivo de 4.5 segundos para no exceder 15 RPM en el Free Tier de Gemini
            time.sleep(4.5)
            return embeddings_model.embed_documents(texts)
        except Exception as e:
            err_str = str(e)
            if "429" in err_str or "RESOURCE_EXHAUSTED" in err_str:
                match = re.search(r"retry in (\d+\.?\d*)s", err_str)
                sleep_time = float(match.group(1)) + 1.0 if match else delay
                print(f"  [429] Límite de cuota en embeddings. Esperando {sleep_time:.2f}s antes de reintentar...")
                time.sleep(sleep_time)
                delay = max(delay * 2, sleep_time)
            else:
                print(f"  Error en embeddings: {e}. Reintentando en {delay}s...")
                time.sleep(delay)
                delay *= 2
    raise Exception("Límite de reintentos superado en la generación de embeddings.")

def index_documents(chunks: list[Document], batch_size: int = 10, delay: float = 2.0) -> int:
    if not chunks:
        return 0

    embeddings_model = get_embeddings()
    total_indexed = 0
    db = SessionLocal()
    
    try:
        for i in range(0, len(chunks), batch_size):
            batch = chunks[i : i + batch_size]
            texts = [chunk.page_content for chunk in batch]
            
            try:
                embeddings = embed_with_retry(embeddings_model, texts)
                db_docs = []
                for chunk, emb in zip(batch, embeddings):
                    db_docs.append(
                        DBDocument(
                            content=chunk.page_content,
                            source=chunk.metadata.get("source_file", chunk.metadata.get("source", "unknown")),
                            page=chunk.metadata.get("page"),
                            chunk_index=chunk.metadata.get("chunk_index"),
                            embedding=emb,
                            metadata_=chunk.metadata
                        )
                    )
                db.add_all(db_docs)
                db.commit()
                total_indexed += len(batch)
            except Exception as e:
                db.rollback()
                print(f"Error indexando batch de chunks en índice {i}: {str(e)}")
                raise e
                
    finally:
        db.close()

    return total_indexed

def index_requirements(records: list[dict], batch_size: int = 10, delay: float = 2.0) -> int:
    if not records:
        return 0
        
    embeddings_model = get_embeddings()
    db = SessionLocal()
    total_indexed = 0
    
    def clean_nan(val, default=None):
        if pd.isna(val):
            return default
        return val

    try:
        for i in range(0, len(records), batch_size):
            batch = records[i : i + batch_size]
            texts = [r["requirement"] for r in batch]
            
            try:
                embeddings = embed_with_retry(embeddings_model, texts)
                db_reqs = []
                for record, emb in zip(batch, embeddings):
                    # Guardar todas las columnas del CSV en el metadato del requisito, limpiando los NaN de Pandas
                    metadata = {
                        "domain": clean_nan(record.get("domain")),
                        "language": clean_nan(record.get("language")),
                        "tags": clean_nan(record.get("tags")),
                        "total": int(record.get("total")) if pd.notna(record.get("total")) else None,
                        "final": clean_nan(record.get("final")),
                        "source": clean_nan(record.get("source"), "csv"),
                        "source_name": clean_nan(record.get("source_name"), "requirements.csv")
                    }
                    
                    db_reqs.append(
                        DBRequirement(
                            text=str(record["requirement"]),
                            source=str(clean_nan(record.get("source"), "csv")),
                            source_name=str(clean_nan(record.get("source_name"), "requirements.csv")),
                            embedding=emb,
                            metadata_=metadata
                        )
                    )
                db.add_all(db_reqs)
                db.commit()
                total_indexed += len(db_reqs)
            except Exception as e:
                db.rollback()
                print(f"Error indexando batch de requisitos en índice {i}: {str(e)}")
                raise e
                
        return total_indexed
    finally:
        db.close()


## 1.5 Limpieza de Requisitos (Opcional)
Ejecuta esta celda si deseas borrar todos los requerimientos existentes en la base de datos antes de cargar el nuevo conjunto.

In [17]:
db = SessionLocal()
try:
    count_before = db.query(DBRequirement).count()
    print(f"Requisitos en base de datos: {count_before}")
    if count_before > 0:
        db.query(DBRequirement).delete()
        db.commit()
        print("¡Base de datos de requisitos limpiada con éxito!")
    else:
        print("La base de datos de requisitos ya está vacía.")
except Exception as e:
    db.rollback()
    print(f"Error al limpiar: {e}")
finally:
    db.close()

Requisitos en base de datos: 714
¡Base de datos de requisitos limpiada con éxito!


## 2. Ingesta de Requerimientos desde CSV
Cada requerimiento se almacena como un único fragmento en la tabla `requirements`.

In [18]:
csv_path = '../data/raw/requirements.csv'

print("Cargando dataset de requerimientos...")
df_reqs = pd.read_csv(csv_path)

# Filtrar requerimientos no nulos y convertir a lista de diccionarios para conservar metadatos
req_records = df_reqs.dropna(subset=['requirement']).to_dict(orient='records')
print(f"Total de requerimientos a indexar: {len(req_records)}")

# Limpiar tabla de requisitos antes de indexar para evitar duplicados
db = SessionLocal()
try:
    db.query(DBRequirement).delete()
    db.commit()
    print("¡Base de datos de requisitos limpiada antes de la carga!")
except Exception as e:
    db.rollback()
    print(f"Error al limpiar tabla: {e}")
finally:
    db.close()

indexed_count = index_requirements(
    records=req_records,
    batch_size=10
)

print(f"Requerimientos indexados exitosamente: {indexed_count}")


Cargando dataset de requerimientos...
Total de requerimientos a indexar: 714
¡Base de datos de requisitos limpiada antes de la carga!
Requerimientos indexados exitosamente: 714


## 3. Preprocesamiento e Ingesta del PDF (Norma ISO)
Se lee el texto del PDF, se elimina ruido (espacios, saltos de página) y se divide en fragmentos de **400 caracteres con 100 de solapamiento**.

In [19]:
pdf_path = '../data/raw/iso_29148.pdf'

def extract_text_from_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"
    return text

print("Extrayendo texto del PDF de la norma ISO...")
if os.path.exists(pdf_path):
    raw_text = extract_text_from_pdf(pdf_path)
    print(f"Caracteres extraídos: {len(raw_text)}")
    
    # Configurar el text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=100,
        separators=["\n\n", "\n", " ", ""]
    )
    
    # Crear chunks
    chunks = text_splitter.split_text(raw_text)
    print(f"Se generaron {len(chunks)} chunks.")
    
    # Convertir a objetos Document de LangChain
    documents = [
        Document(
            page_content=chunk, 
            metadata={"source": "pdf", "source_file": "iso_29148.pdf", "chunk_index": i}
        ) 
        for i, chunk in enumerate(chunks)
    ]
    
    # Indexar documentos en la base de datos vectorial
    print("Indexando chunks en la base de datos...")
    docs_indexed = index_documents(documents)
    print(f"Chunks indexados exitosamente: {docs_indexed}")
else:
    print(f"No se encontró el archivo: {pdf_path}")

Extrayendo texto del PDF de la norma ISO...
Caracteres extraídos: 327211
Se generaron 1080 chunks.
Indexando chunks en la base de datos...
Chunks indexados exitosamente: 1080
